# Translator Variance in a Meaning-Controlled Corpus

**Purpose.** This tests the central premise of the paper: that
a fixed source text translated into many languages holds *meaning* constant, so that any
quantity varying across languages reflects language, script or model rather than content.
The earlier evidence was two Bosnian translations differing by 0.4% in bytes per verse, a single
Latin-script pair that cannot rule out larger variation elsewhere.

**What this notebook does.** It measures translator variance directly, across several scripts, and
reports it against the between-language spread the paper attributes to language.

| Language | Script | Translations available |
|---|---|---|
| English | Latin | rwwad, saheeh, hilali_khan |
| Turkish | Latin | rwwad, shaban, shahin |
| Spanish | Latin | montada_eu, garcia, montada_latin |
| Indonesian | Latin | complex, sabiq, affairs |
| French / German / Albanian / Uzbek / Bosnian | Latin | 2 each |
| **Chinese** | **CJK** | suliman, makin |
| **Khmer** | **Khmer** | rwwad, cambodia |
| **Tamil** | **Brahmic** | omar_brief, omar, baqavi |

The three non-Latin languages are the point. If translator variance is small in Khmer and Tamil
too, the identification strategy is validated on evidence instead of assumption.

**Three questions:**
1. **Fertility** — how much do two translations of the same language differ in tokens per verse,
   against how much languages differ from each other?
2. **Meaning** — are two translations of the same language closer in embedding space than a
   translation is to its counterpart in another language?
3. **Retrieval** — would swapping translator A for translator B change the paper's retrieval numbers?

**Runtime.** Cells 1–6 are CPU-only, about 25 minutes, mostly downloading. Cell 7 onward needs a
GPU (Runtime → Change runtime type → T4). Total under an hour on a free T4.

**Outputs.** Six CSVs and a summary, written to `results_translator_variance/`, plus a zip to download.


## 1. Setup

In [ ]:
!pip -q install transformers==4.44.2 tiktoken sentencepiece sentence-transformers==3.0.1 protobuf --upgrade
import os, json, time, itertools, warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, requests

OUT = 'results_translator_variance'
CACHE = 'quranenc_cache'
os.makedirs(OUT, exist_ok=True); os.makedirs(CACHE, exist_ok=True)
print('ready')

## 2. Which languages have more than one translation?

`/api/v1/translations/list` returns everything QuranEnc hosts. We keep languages with at least two
independent translations, and mark which of them are in the paper's 41-language corpus.

In [ ]:
LIST_URL = 'https://quranenc.com/api/v1/translations/list'

r = requests.get(LIST_URL, timeout=60); r.raise_for_status()
raw = r.json()

# The payload nests keys by language group; flatten defensively rather than assuming a shape.
def flatten_keys(obj, acc=None):
    acc = [] if acc is None else acc
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k in ('key', 'translation_key') and isinstance(v, str): acc.append(v)
            else: flatten_keys(v, acc)
    elif isinstance(obj, list):
        for v in obj: flatten_keys(v, acc)
    return acc

keys = sorted(set(flatten_keys(raw)))
print(f'{len(keys)} translation keys found')

# Fallback: the list endpoint has changed shape before. Hard-code what we verified on 16 Sep 2026.
if len(keys) < 20:
    keys = ('english_rwwad english_saheeh english_hilali_khan french_rashid french_montada '
            'spanish_garcia spanish_montada_eu spanish_montada_latin german_rwwad german_bubenheim '
            'turkish_rwwad turkish_shaban turkish_shahin albanian_rwwad albanian_nahi '
            'bosnian_rwwad bosnian_mihanovich uzbek_rwwad uzbek_mansour '
            'indonesian_sabiq indonesian_affairs indonesian_complex chinese_suliman chinese_makin '
            'khmer_rwwad khmer_cambodia tamil_omar_brief tamil_omar tamil_baqavi '
            'swahili_rwwad swahili_barawani amharic_zain amharic_sadiq').split()
    print('using verified fallback list:', len(keys))

lang_of = lambda k: k.split('_')[0]
groups = {}
for k in keys: groups.setdefault(lang_of(k), []).append(k)
multi = {l: sorted(v) for l, v in groups.items() if len(v) >= 2}

# Script labels; the non-Latin languages are what make this study informative.
SCRIPT = {'english':'Latin','french':'Latin','spanish':'Latin','german':'Latin','turkish':'Latin',
          'albanian':'Latin','bosnian':'Latin','uzbek':'Latin','indonesian':'Latin','swahili':'Latin',
          'chinese':'CJK','khmer':'Khmer','tamil':'Brahmic','amharic':'Ethiopic'}
IN_CORPUS = {'english','french','spanish','german','turkish','albanian','bosnian','uzbek',
             'indonesian','chinese','khmer','tamil'}

multi = {l: v for l, v in multi.items() if l in SCRIPT}
print(f'\n{len(multi)} languages with 2+ translations:')
for l, v in sorted(multi.items(), key=lambda x: (SCRIPT[x[0]], x[0])):
    tag = 'in corpus' if l in IN_CORPUS else 'extra'
    print(f'  {l:12s} {SCRIPT[l]:9s} ({tag:9s}) {len(v)}: {", ".join(v)}')

## 3. Download

One request per sura per translation (114 each), cached to disk so a re-run is free.
Verse counts are checked against the canonical Hafs total of 6,236 — a translation that
does not align exactly is dropped rather than silently padded.

In [ ]:
SURA_URL = 'https://quranenc.com/api/v1/translation/sura/{key}/{sura}'
HAFS_TOTAL = 6236

def fetch_translation(key, sleep=0.15, retries=3):
    path = f'{CACHE}/{key}.json'
    if os.path.exists(path):
        return pd.read_json(path)
    rows = []
    for sura in range(1, 115):
        for attempt in range(retries):
            try:
                resp = requests.get(SURA_URL.format(key=key, sura=sura), timeout=60)
                resp.raise_for_status()
                rows.extend(resp.json()['result']); break
            except Exception as e:
                if attempt == retries - 1:
                    print(f'  ! {key} sura {sura}: {e}'); return None
                time.sleep(2 ** attempt)
        time.sleep(sleep)
    df = pd.DataFrame(rows)
    df.to_json(path, orient='records', force_ascii=False)
    return df

texts = {}
for lang, klist in sorted(multi.items()):
    for key in klist:
        print(f'{key} ...', end=' ', flush=True)
        df = fetch_translation(key)
        if df is None: print('FAILED'); continue
        df = df[['sura', 'aya', 'translation']].copy()
        df['sura'] = df.sura.astype(int); df['aya'] = df.aya.astype(int)
        df = df.drop_duplicates(['sura', 'aya']).sort_values(['sura', 'aya'])
        if len(df) != HAFS_TOTAL:
            print(f'SKIP ({len(df)} verses, expected {HAFS_TOTAL})'); continue
        texts[key] = df.reset_index(drop=True)
        print(f'ok ({len(df)})')

print(f'\n{len(texts)} translations aligned at {HAFS_TOTAL} verses')
langs_ok = {l: [k for k in v if k in texts] for l, v in multi.items()}
langs_ok = {l: v for l, v in langs_ok.items() if len(v) >= 2}
print('usable languages:', {l: len(v) for l, v in sorted(langs_ok.items())})

## 4. Verse-level alignment across translations

Before comparing anything, confirm the (sura, aya) keys are identical across every translation,
the same check the paper applies to the 41-file corpus.

In [ ]:
keysets = {k: set(zip(v.sura, v.aya)) for k, v in texts.items()}
inter = set.intersection(*keysets.values()); union = set.union(*keysets.values())
print(f'union {len(union)}  intersection {len(inter)}  identical: {len(union) == len(inter) == HAFS_TOTAL}')

# align all to one verse order
order = sorted(inter)
for k in texts:
    t = texts[k].set_index(['sura', 'aya']).loc[order].reset_index()
    texts[k] = t
print('all translations aligned to a common verse order')

## 5. Fertility by translation

The eleven production tokenizers from the paper's Appendix D, unchanged, including the ungated
mirror checkpoints for Llama-3 and Gemma-2.

In [ ]:
import tiktoken
from transformers import AutoTokenizer

HF = {
 'mbert':       'bert-base-multilingual-cased',
 'xlmr':        'xlm-roberta-base',
 'bloom':       'bigscience/bloom-560m',
 'nllb':        'facebook/nllb-200-distilled-600M',
 'mt5':         'google/mt5-small',
 'byt5':        'google/byt5-small',
 'llama3':      'unsloth/llama-3-8b',
 'gemma':       'unsloth/gemma-2-2b',
}
tok_hf = {}
for name, ckpt in HF.items():
    print('loading', name, '...', end=' ', flush=True)
    try:
        tok_hf[name] = AutoTokenizer.from_pretrained(ckpt, use_fast=True); print('ok')
    except Exception as e:
        print('FAILED', e)
enc_o200k = tiktoken.get_encoding('o200k_base')
enc_cl100k = tiktoken.get_encoding('cl100k_base')
print('tokenizers ready:', len(tok_hf) + 2)

In [ ]:
def count_tokens(lines, name):
    if name == 'gpt4o_o200k':  return np.array([len(x) for x in enc_o200k.encode_ordinary_batch(lines)])
    if name == 'gpt4_cl100k':  return np.array([len(x) for x in enc_cl100k.encode_ordinary_batch(lines)])
    t = tok_hf[name]
    out = []
    B = 512
    for i in range(0, len(lines), B):
        enc = t(lines[i:i+B], add_special_tokens=False)['input_ids']
        out.extend(len(e) for e in enc)
    return np.array(out)

TOK_NAMES = ['gpt4o_o200k', 'gpt4_cl100k'] + list(tok_hf)

rows = []
for lang, klist in sorted(langs_ok.items()):
    for key in klist:
        lines = texts[key].translation.fillna('').astype(str).tolist()
        rec = {'language': lang, 'script': SCRIPT[lang], 'translation_key': key,
               'in_corpus': lang in IN_CORPUS,
               'bytes_per_verse': np.mean([len(s.encode('utf-8')) for s in lines]),
               'chars_per_verse': np.mean([len(s) for s in lines]),
               'words_per_verse': np.mean([len(s.split()) for s in lines])}
        for name in TOK_NAMES:
            rec[f'{name}_tpv'] = float(count_tokens(lines, name).mean())
        rows.append(rec)
        print(f'{key:24s} bytes/verse {rec["bytes_per_verse"]:7.1f}  gpt4o {rec["gpt4o_o200k_tpv"]:7.1f}')

fert = pd.DataFrame(rows)
fert.to_csv(f'{OUT}/fertility_by_translation.csv', index=False)
fert.head(20)

## 6. The headline number: within-language vs between-language

Two quantities. **Within-language spread** is the max/min ratio across translations of the same
language. **Between-language spread** is the p90/p10 ratio the paper reports. The claim under
challenge is that the first is negligible against the second.

The variance-components model makes it one number: the intraclass correlation, the share of total
fertility variance sitting *between* languages rather than between translators within a language.
An ICC near 1 means translator choice is noise against the language effect.

In [ ]:
def icc_between_language(values, langs):
    "Share of variance between languages (one-way random effects)."
    df = pd.DataFrame({'y': values, 'lang': langs})
    k = df.groupby('lang').size()
    grand = df.y.mean()
    msb = sum(n * (g.y.mean() - grand) ** 2 for (l, g), n in zip(df.groupby('lang'), k)) / (len(k) - 1)
    msw = sum(((g.y - g.y.mean()) ** 2).sum() for _, g in df.groupby('lang')) / (len(df) - len(k))
    k0 = (len(df) - (k ** 2).sum() / len(df)) / (len(k) - 1)
    var_b = max((msb - msw) / k0, 0.0)
    return var_b / (var_b + msw) if (var_b + msw) > 0 else np.nan

summary = []
for name in ['bytes_per_verse'] + [f'{n}_tpv' for n in TOK_NAMES]:
    within = fert.groupby('language')[name].agg(lambda s: s.max() / s.min())
    between = fert.groupby('language')[name].mean()
    p90p10 = np.percentile(between, 90) / np.percentile(between, 10)
    summary.append({
        'measure': name,
        'within_max_over_min_median': round(float(within.median()), 4),
        'within_max_over_min_worst': round(float(within.max()), 4),
        'worst_language': within.idxmax(),
        'between_language_p90_p10': round(float(p90p10), 3),
        'icc_between_language': round(float(icc_between_language(fert[name].values, fert.language.values)), 4),
    })
summ = pd.DataFrame(summary)
summ['between_over_within'] = ((summ.between_language_p90_p10 - 1) /
                               (summ.within_max_over_min_median - 1).replace(0, np.nan)).round(1)
summ.to_csv(f'{OUT}/within_vs_between_summary.csv', index=False)
print(summ.to_string(index=False))

print('\nPer-language within-translator spread (GPT-4o tokens/verse), by script:')
wl = (fert.groupby(['script', 'language'])
          .agg(n_translations=('translation_key', 'size'),
               gpt4o_min=('gpt4o_o200k_tpv', 'min'), gpt4o_max=('gpt4o_o200k_tpv', 'max'),
               bytes_min=('bytes_per_verse', 'min'), bytes_max=('bytes_per_verse', 'max')))
wl['gpt4o_ratio'] = (wl.gpt4o_max / wl.gpt4o_min).round(4)
wl['bytes_ratio'] = (wl.bytes_max / wl.bytes_min).round(4)
wl = wl.sort_values('gpt4o_ratio', ascending=False)
wl.to_csv(f'{OUT}/within_language_by_script.csv')
print(wl[['n_translations', 'gpt4o_ratio', 'bytes_ratio']].to_string())

### What to conclude

- **ICC above ~0.95 for every tokenizer** → translator choice is negligible; the paper's premise
  holds and can be stated as measured rather than assumed.
- **Non-Latin ratios comparable to Latin ones** → the Bosnian result was not a Latin-script artifact,
  which is precisely what needs testing.
- **Any language whose within-ratio approaches the between-language spread** → report it. A negative
  result here is still a contribution: it would be the first measurement of how much "meaning-controlled"
  corpora actually control meaning.

## 7. Does translator choice change *meaning*? (GPU)

Fertility is about surface form. The stronger version of the objection is semantic:
different translators may render different content. Test it directly — embed every verse under
each translation and ask whether two translations of the same language are closer to each other
than either is to a different language's translation of the same verse.

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEV)
if DEV == 'cpu':
    print('WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU, then re-run from here.')

SAMPLE = 1000
rng = np.random.default_rng(42)
idx = np.sort(rng.choice(HAFS_TOTAL, SAMPLE, replace=False))

model = SentenceTransformer('sentence-transformers/LaBSE', device=DEV)

emb = {}
for key in texts:
    lines = texts[key].translation.fillna('').astype(str).iloc[idx].tolist()
    emb[key] = model.encode(lines, batch_size=64, convert_to_numpy=True,
                            normalize_embeddings=True, show_progress_bar=False)
    print('embedded', key)

In [ ]:
# same verse, same language, different translator  vs  same verse, different language
rows = []
for lang, klist in sorted(langs_ok.items()):
    for a, b in itertools.combinations(klist, 2):
        within = float(np.mean(np.sum(emb[a] * emb[b], axis=1)))
        others = [l for l in langs_ok if l != lang]
        cross = float(np.mean([np.mean(np.sum(emb[a] * emb[langs_ok[o][0]], axis=1)) for o in others]))
        rows.append({'language': lang, 'script': SCRIPT[lang], 'pair': f'{a} | {b}',
                     'cos_same_lang_diff_translator': round(within, 4),
                     'cos_diff_lang': round(cross, 4),
                     'gap': round(within - cross, 4)})
sem = pd.DataFrame(rows)
sem.to_csv(f'{OUT}/semantic_translator_variance.csv', index=False)
print(sem.to_string(index=False))
print('\nmean gap by script:')
print(sem.groupby('script').gap.agg(['mean', 'min', 'count']).round(4).to_string())

## 8. Would swapping translators change the paper's retrieval numbers? (GPU)

The practical question. Re-run English-paired retrieval using each alternative translation in turn.
If per-language top-1 moves by less than the between-language spread, the paper's retrieval
conclusions do not depend on which translator was picked.

In [ ]:
EN_REF = 'english_rwwad'    # the paper's English file
assert EN_REF in emb, 'English reference missing'

def top1(src, tgt):
    sim = src @ tgt.T
    return float((np.argmax(sim, axis=1) == np.arange(len(src))).mean())

rows = []
for lang, klist in sorted(langs_ok.items()):
    if lang == 'english': continue
    for key in klist:
        a = top1(emb[key], emb[EN_REF])      # X -> English
        b = top1(emb[EN_REF], emb[key])      # English -> X
        rows.append({'language': lang, 'script': SCRIPT[lang], 'translation_key': key,
                     'top1_to_en': round(a, 4), 'top1_from_en': round(b, 4),
                     'top1_mean': round((a + b) / 2, 4)})
ret = pd.DataFrame(rows)
ret.to_csv(f'{OUT}/retrieval_by_translation.csv', index=False)

spread = ret.groupby('language').top1_mean.agg(['min', 'max', 'size'])
spread['within_range'] = (spread['max'] - spread['min']).round(4)
between = ret.groupby('language').top1_mean.mean()
print(ret.to_string(index=False))
print('\nwithin-language retrieval range vs between-language range:')
print(spread.to_string())
print(f'\nbetween-language range: {between.max() - between.min():.4f}')
print(f'largest within-language range: {spread.within_range.max():.4f} '
      f'({spread.within_range.idxmax()})')
spread.to_csv(f'{OUT}/retrieval_within_language_spread.csv')

## 9. Package the results

In [ ]:
verdict = {
    'n_languages': len(langs_ok),
    'n_translations': len(texts),
    'scripts': sorted({SCRIPT[l] for l in langs_ok}),
    'icc_min_across_tokenizers': float(summ.icc_between_language.min()),
    'icc_median': float(summ.icc_between_language.median()),
    'worst_within_language_fertility_ratio': float(summ.within_max_over_min_worst.max()),
    'median_between_over_within': float(summ.between_over_within.median()),
}
try:
    verdict['min_semantic_gap'] = float(sem.gap.min())
    verdict['max_within_language_retrieval_range'] = float(spread.within_range.max())
except NameError:
    pass
with open(f'{OUT}/verdict.json', 'w') as f: json.dump(verdict, f, indent=2)
print(json.dumps(verdict, indent=2))

!zip -qr results_translator_variance.zip {OUT} && echo "zipped"
try:
    from google.colab import files; files.download('results_translator_variance.zip')
except Exception as e:
    print('download manually from the file browser:', e)